In [1]:
import sys
sys.path.insert(0, './models')
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = '1'
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from train import train, get_time_dif
import importlib
import argparse
import warnings
warnings.filterwarnings("ignore")

In [2]:
from torch.utils.tensorboard import SummaryWriter
from sklearn import metrics
import time
from datetime import timedelta
from collections import defaultdict
import json
import os
import traceback


In [3]:
def get_time_dif(start_time):
    """获取已使用时间"""
    end_time = time.time()
    time_dif = end_time - start_time
    return timedelta(seconds=int(round(time_dif)))

In [4]:
def train(config, model, dataset):
    logging = config.logging
    start_time = time.time()
    parameters = filter(lambda x: x.requires_grad, model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=config.learning_rate, weight_decay=1e-4)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.6)
    total_batch = 0  
    val_best_auc = -np.inf
    val_best_loss = np.inf
    last_improve = 0  
    flag1 = False
    no_improve_cnt = 0
    model.train()
    writer = SummaryWriter(log_dir=config.summary_dir)
    for epoch in range(config.num_epochs):
        logging.info('Epoch [{}/{}]'.format(epoch + 1, config.num_epochs))
        logging.info('start init data...')
        dataset.init_per_epoch()
        logging.info('init data done.')
        for batch_datas in dataset.train_iter:
            batch_Y = batch_datas[-1]
            out, loss = model(batch_datas)
            if config.multi_gpu:
                loss = loss.sum()
            model.zero_grad()
            loss.backward()
            optimizer.step()
            if total_batch > 0 and total_batch % config.batches_per_check == 0:
                logging.info('step: %s, start eval...', total_batch)
                true = batch_Y.data.cpu()
                predict = torch.max(out.data, 1)[1].cpu()
                train_loss = loss.item()
                train_acc = metrics.accuracy_score(true, predict)
                val_acc, val_auc, val_loss = evaluate(config, model, dataset.val_iter)
                if val_best_auc < val_auc:
                    val_best_auc = val_auc
                    save_checkpoint(model, config)
                    # torch.save(model.state_dict(), config.save_path)
                    improve = '***'
                    last_improve = total_batch
                    no_improve_cnt = 0
                else:
                    improve = ''
                    no_improve_cnt += 1
                time_dif = get_time_dif(start_time)
                lr = optimizer.param_groups[0]['lr']
                msg = 'Iter: {0:>6},  Train Loss: {1:>5.3},  Train Acc: {2:>6.2%},  Val Loss: {3:>5.3},  Val Acc: {4:>6.2%},  Val Auc: {5:>6.4}, Lr: {6:>8.6}, Time: {7} {8}'
                msg = msg.format(total_batch, train_loss, train_acc, val_loss, val_acc, val_auc, lr, time_dif, improve)
                logging.info(msg)
                model.train()
                writer.add_scalar('train_loss', train_loss, total_batch)
                writer.add_scalar('train_acc', train_acc, total_batch)
                writer.add_scalar('val_loss', val_loss, total_batch)
                writer.add_scalar('val_auc', val_auc, total_batch)
                writer.add_scalar('val_acc', val_acc, total_batch)
                writer.add_scalar('lr', lr, total_batch)
                if no_improve_cnt > 0 and no_improve_cnt % 3 == 0 and lr > 3e-5:
                    lr_scheduler.step()
            if total_batch - last_improve > config.require_improvement:
                logging.info("No optimization for a long time, auto-stopping...")
                flag1 = True
                break
            total_batch += 1
        if flag1:
            break
        test(config, model, dataset.test_iter)  # 每个epoch eval一次
    writer.close()
    test(config, model, dataset.test_iter)

In [5]:
def save_checkpoint(model, config):
    ckpt_dict = {
        'model': model.module.state_dict() if config.multi_gpu else model.state_dict(),
        'config': config.get_parameters()
    }
    torch.save(ckpt_dict, config.save_path)

In [6]:
def load_checkpoint(model, config):
    ckpt_dict = torch.load(config.save_path)
    if config.multi_gpu:
        model.module.load_state_dict(ckpt_dict['model'])
    else:
        model.load_state_dict(ckpt_dict['model'])

In [7]:
def test(config, model, test_iter):
    logging = config.logging
    # test
    # model.load_state_dict(torch.load(config.save_path))
    if not os.path.exists(config.save_path):
        logging.info('model path does not exist...')
        return
    load_checkpoint(model, config)
    model.eval()
    start_time = time.time()
    test_acc, test_auc, loss_total, test_loss, test_report, test_confusion, pred_prob_all, labels_all = evaluate(config, model, test_iter, test=True)
    # compute new_user/old_user/new_news/old_news auc
    try:
        new_user_pred = [x for i, x in enumerate(pred_prob_all) if i in config.new_user_index]
        new_user_label = [x for i, x in enumerate(labels_all) if i in config.new_user_index]
        new_user_auc = metrics.roc_auc_score(new_user_label, new_user_pred)

        old_user_pred = [x for i, x in enumerate(pred_prob_all) if i not in config.new_user_index]
        old_user_label = [x for i, x in enumerate(labels_all) if i not in config.new_user_index]
        old_user_auc = metrics.roc_auc_score(old_user_label, old_user_pred)
        
        new_news_pred = [x for i, x in enumerate(pred_prob_all) if i in config.new_news_index]
        new_news_label = [x for i, x in enumerate(labels_all) if i in config.new_news_index]
        new_news_auc = metrics.roc_auc_score(new_news_label, new_news_pred)
        
        old_news_pred = [x for i, x in enumerate(pred_prob_all) if i not in config.new_news_index]
        old_news_label = [x for i, x in enumerate(labels_all) if i not in config.new_news_index]
        old_news_auc = metrics.roc_auc_score(old_news_label, old_news_pred)   
        
        rig = 1 - loss_total / config.entropy  # 信息增益
        gauc = cal_gauc(config.test_user_id, pred_prob_all, labels_all)
        
        pred_label = {
            'pred': [str(round(x, 3)) for x in pred_prob_all],
            'label': list(map(str, labels_all))
        }
        msg = 'Test Loss: {0:>5.2}, Test Acc: {1:>6.2%}, Test Auc: {2:>6.4}, new_user Auc: {3:>6.4}, old_user Auc: {4:>6.4}, new_news Auc: {5:>6.4}, old_news Auc: {6:>6.4}, total_loss: {7:>7.4}, rig: {8:>8.4}, gauc: {9:>9.4}'
        msg = msg.format(test_loss, test_acc, test_auc, new_user_auc, old_user_auc, new_news_auc, old_news_auc, loss_total, rig, gauc)
        logging.info(msg.format())
        logging.info("Precision, Recall and F1-Score...")
        logging.info(test_report)
        logging.info("Confusion Matrix...")
        logging.info(test_confusion)
        time_dif = get_time_dif(start_time)
        logging.info("Time usage: %s", time_dif)
        model.train()
        with open(config.pred_label_save_path, 'w') as f:
            json.dump(pred_label, f)
    except Exception as e:
        traceback.print_exc()

In [8]:
def evaluate(config, model, data_iter, test=False):
    model.eval()
    loss_total = 0
    predict_all = []
    labels_all = []
    pred_prob_all = []
    with torch.no_grad():
        for batch_datas in data_iter:
            batch_Y = batch_datas[-1]
            out, loss = model(batch_datas)
            if config.multi_gpu:
                loss = loss.sum()
            # loss = F.cross_entropy(out, batch_Y)
            loss_total += loss.item()
            labels = batch_Y.data.cpu().numpy()
            predict = torch.max(out.data, 1)[1].cpu().numpy()
            predict_all.append(predict)
            labels_all.append(labels)
            pred_prob = F.softmax(out, dim=1)[:, 1].data.cpu().numpy()
            pred_prob_all.append(pred_prob)
    predict_all = np.concatenate(predict_all)
    labels_all = np.concatenate(labels_all)
    pred_prob_all = np.concatenate(pred_prob_all)
    acc = metrics.accuracy_score(labels_all, predict_all)
    try:
        auc = metrics.roc_auc_score(labels_all, pred_prob_all)
    except Exception as e:
        import pdb; pdb.set_trace()
    if test:
        report = metrics.classification_report(labels_all, predict_all, target_names=config.class_list, digits=4)
        confusion = metrics.confusion_matrix(labels_all, predict_all)
        return acc, auc, loss_total, loss_total / len(data_iter), report, confusion, pred_prob_all, labels_all
    return acc, auc, loss_total / len(data_iter)

In [9]:
def cal_gauc(user_ids, preds, labels):
    assert len(user_ids) == len(preds) and len(preds) == len(labels)
    uid_pred_dict = defaultdict(list)
    uid_label_dict = defaultdict(list)
    for uid, pred, label in zip(user_ids, preds, labels):
        uid_pred_dict[uid].append(pred)
        uid_label_dict[uid].append(label)
    all_auc = 0
    total = 0
    for uid in uid_pred_dict.keys():
        if len(set(uid_label_dict[uid])) < 2:
            continue
        u_auc = metrics.roc_auc_score(uid_label_dict[uid], uid_pred_dict[uid])
        n_u = len(uid_label_dict[uid])
        all_auc += u_auc * n_u
        total += n_u  
    return all_auc / total

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import numpy as np
import time
import logging
import sys


class Config(object):
    def __init__(self, args):
        self.model_name = 'MPNRec'
        self.train_behavior_path = './data/behavior_sample_train.tsv'
        self.val_behavior_path = './data/behavior_sample_val.tsv'
        self.news_path = './data/all_news.tsv'
        self.news2idx_path = './data/news2idx.json'
        self.uid2idx_path = './data/uid2idx.json'
        self.cate2idx_path = './data/cate2idx.json'
        self.vocab_path = './data/vocab.json'
        self.uid_embedding_deepwalk_path = './data/uid_embeddings_deepwalk.npy'
        self.news_embeddings_deepwalk_path = './data/news_embeddings_deepwalk.npy'
        self.news_bert_embedding_path = './data/news_bert_embedding.npy'
        self.w2v_embedding_path = './data/w2v_embedding.npy'
        self.cate2news_path = './data/cate2news.json'
        self.topic2news_path = './data/topic2news.json'
        self.news2uid_path = './data/news2uid.json'
        self.new_user_path = './data/new_user.csv'
        self.new_news_path = './data/new_news.csv'
        self.text_encoding = args.text_encoding
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.debug = args.debug
        self.device_count = torch.cuda.device_count()
        self.multi_gpu = self.device_count > 1 and args.multi_gpu
        self.text_len = args.text_len # default 512
        self.learning_rate = args.lr
        self.sample_size = args.sample_size
        self.history_len = args.history_len
        self.batch_size = args.batch_size
        self.neg_pos_ratio = args.neg_pos_ratio
        self.num_epochs = args.num_epochs
        self.batches_per_check = args.batches_per_check
        self.require_improvement = args.require_improvement
        self.use_pretrain = args.use_pretrain
        self.agg_method = args.agg_method # pooling or self-attention
        self.reinit_epoch = args.reinit_epoch
        self.embedding_dim = 128
        self.context_emb_dim = 16
        self.uid_num = -1
        self.nid_num = -1
        self.cate_num = -1
        self.topic_num = 201
        self.vocab_size = -1
        trial_name = 'debug_{}_prefix_{}_sample_size_{}_batch_size_{}_text_encoding_{}_history_len_{}_pretrain_{}_agg_method_{}_lr_{}_neg_pos_ratio_{}_epochs_{}_batches_per_check_{}_text_len_{}'.format(
            self.debug,
            args.prefix,
            self.sample_size,
            self.batch_size,
            self.text_encoding,
            self.history_len,
            self.use_pretrain,
            self.agg_method,
            self.learning_rate,
            self.neg_pos_ratio,
            self.num_epochs,
            self.batches_per_check,
            self.text_len
        )
        self.summary_dir = './models/{}/summary/{}'.format(self.model_name, trial_name)
        self.save_path = './models/{}/checkpoint/{}.ckpt'.format(self.model_name, trial_name)
        self.log_path = './models/{}/logs/{}.log'.format(self.model_name, trial_name)
        self.pred_label_save_path = './models/{}/predict/{}.json'.format(self.model_name, trial_name)
        self.class_list = ['0', '1']
        if self.debug:
            logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)-15s %(message)s')
        else:
            logging.basicConfig(level=logging.INFO, filename=self.log_path, filemode='w', format='%(asctime)-15s %(message)s')
        self.logging = logging
        
    def get_parameters(self):
        return {
            'sample_size': self.sample_size,
            'learning_rate': self.learning_rate,
            'history_len': self.history_len,
            'batch_size': self.batch_size,
            'neg_pos_ratio': self.neg_pos_ratio,
            'num_epochs': self.num_epochs,
            'batches_per_check': self.batches_per_check,
            'require_improvement': self.require_improvement,
            'text_encoding': self.text_encoding,
            'text_len': self.text_len,
            'agg_method': self.agg_method,
            'use_pretrain': self.use_pretrain
        }

class Model(nn.Module):
    def __init__(self, config):
        super(Model, self).__init__()
        self.config = config
        self.S_tensor = nn.Parameter(torch.arange(config.sample_size), requires_grad=False)
        self.L_tensor = nn.Parameter(torch.arange(config.history_len), requires_grad=False)
        if config.use_pretrain:
            self.load_pretrain_embeddings(config)
        else:
            self.uid_embedding = nn.Embedding(config.uid_num, config.embedding_dim)
            self.nid_embedding_structual = nn.Embedding(config.nid_num, config.embedding_dim)
            self.nid_embedding_semantic_pre = nn.Embedding.from_pretrained(torch.FloatTensor(np.load(config.news_bert_embedding_path)), freeze=True)
        self.category_embedding = nn.Embedding(config.cate_num, config.embedding_dim)
        self.topic_embedding = nn.Embedding(config.topic_num, config.embedding_dim)
        self.hour_embedding = nn.Embedding(24, config.context_emb_dim)
        self.nid_proj = nn.Linear(768, config.embedding_dim)  # bert embedding 投影
        # self.news_cate_news_att = nn.Sequential(
        #     nn.Linear(3 * config.embedding_dim, 128),
        #     nn.LeakyReLU(inplace=True),
        #     nn.Linear(128, 3)
        # )
        # self.news_topic_news_att = nn.Sequential(
        #     nn.Linear(3 * config.embedding_dim, 128),
        #     nn.LeakyReLU(inplace=True),
        #     nn.Linear(128, 3)
        # )
        # self.news_user_att = nn.Sequential(
        #     nn.Linear(2 * config.embedding_dim, 128),
        #     nn.LeakyReLU(inplace=True),
        #     nn.Linear(128, 2)
        # )
        # self.meta_path_attention = nn.Sequential(
        #     nn.Linear(config.embedding_dim * 3, 128),
        #     nn.LeakyReLU(inplace=True),
        #     nn.Linear(128, 3)
        # )
        # self.din_attention = nn.Sequential(
        #     nn.Linear(config.embedding_dim * 6, 128),
        #     nn.LeakyReLU(inplace=True),
        #     nn.Linear(128, 1)
        # )
        fc_input_dim = 12 * config.embedding_dim + config.context_emb_dim + 3
        self.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(fc_input_dim, 512),
            nn.LeakyReLU(inplace=True),
            nn.Linear(512, 128),
            nn.LeakyReLU(inplace=True),
            nn.Linear(128, 2)
        )
        self.init_network()
        
    # 权重初始化，默认xavier
    def init_network(self, method='xavier', exclude='embedding', seed=123):
        for name, w in self.named_parameters():
            if exclude not in name:
                if len(w.size()) < 2:
                    continue
                if 'weight' in name:
                    if method == 'xavier':
                        nn.init.xavier_normal_(w)
                    elif method == 'kaiming':
                        nn.init.kaiming_normal_(w)
                    else:
                        nn.init.normal_(w)
                elif 'bias' in name:
                    nn.init.constant_(w, 0)
                else:
                    pass
    
    def load_pretrain_embeddings(self, config):
        self.uid_embedding = nn.Embedding.from_pretrained(torch.FloatTensor(np.load(config.uid_embedding_deepwalk_path)), freeze=False)
        self.nid_embedding_structual = nn.Embedding.from_pretrained(torch.FloatTensor(np.load(config.news_embeddings_deepwalk_path)), freeze=False)
        self.nid_embedding_semantic_pre = nn.Embedding.from_pretrained(torch.FloatTensor(np.load(config.news_bert_embedding_path)), freeze=True)
    
    def att_agg_user(self, uid_embedding, sample_embedding, sample_len):
        """@params
        uid_embedding: [B, D]
        sample_embedding: [B, L, S, D] batch_size, seq_len, sample_size, embedding_dim
        sample_len: [B, L]
        returns: [B, L, D]
        """
        B, L, S, D = sample_embedding.size()
        att_raw_score = torch.multiply(uid_embedding.view(B, 1, 1, D), sample_embedding)
        mask = (self.S_tensor.view(1, 1, -1) >=  sample_len.view(B, L, 1)).view(B, L, S, 1)  # B, L, S, 1
        att_raw_score = att_raw_score.masked_fill(mask, -1e9)
        att_score = F.softmax(att_raw_score, dim=2)
        return torch.sum(att_score * sample_embedding, dim=2)
    
    def att_agg_news(self, nid_embedding, sample_embedding, sample_len):
        """
        nid_embedding: [B, D]
        sample_embedding: [B, S, D]
        sample_len: [B]
        """
        B, S, D = sample_embedding.size()
        att_raw_score = torch.multiply(nid_embedding.view(B, 1, D), sample_embedding)
        mask = (self.S_tensor.view(1, -1) >=  sample_len.view(B, 1)).view(B, S, 1)  # B, S, 1
        att_raw_score = att_raw_score.masked_fill(mask, -1e9)
        att_score = F.softmax(att_raw_score, dim=1)
        return torch.sum(att_score * sample_embedding, dim=1)
    
    # def news_x_news_att(self, news_embedding_1, x_embedding, news_embedding_2, att_f):
    #     if len(news_embedding_1.size()) == 3:
    #         B, L, D = news_embedding_1.size()
    #         news_X_news_input = torch.cat([news_embedding_1, x_embedding, news_embedding_2], dim=-1)
    #         news_X_news_score = att_f(news_X_news_input)
    #         news_X_news_score = F.softmax(news_X_news_score, dim=-1).view(B, L, 3, 1)
    #         return torch.sum(news_X_news_score * news_X_news_input.view(B, L, 3, D), dim=2)
    #     else:
    #         B, D = news_embedding_1.size()
    #         news_X_news_input = torch.cat([news_embedding_1, x_embedding, news_embedding_2], dim=-1)
    #         news_X_news_score = att_f(news_X_news_input)
    #         news_X_news_score = F.softmax(news_X_news_score, dim=-1).view(B, 3, 1)
    #         return torch.sum(news_X_news_score * news_X_news_input.view(B, 3, D), dim=1)
        
    # def news_X_att(self, news_embedding, user_embedding, att_f):
    #     if len(news_embedding.size()) == 3:
    #         B, L, D = news_embedding.size()
    #         news_X_news_input = torch.cat([news_embedding, user_embedding], dim=-1)
    #         news_X_news_score = att_f(news_X_news_input)
    #         news_X_news_score = F.softmax(news_X_news_score, dim=-1).view(B, L, 2, 1)
    #         return torch.sum(news_X_news_score * news_X_news_input.view(B, L, 2, D), dim=2)
    #     else:
    #         B, D = news_embedding.size()
    #         news_X_news_input = torch.cat([news_embedding, user_embedding], dim=-1)
    #         news_X_news_score = att_f(news_X_news_input)
    #         news_X_news_score = F.softmax(news_X_news_score, dim=-1).view(B, 2, 1)
    #         return torch.sum(news_X_news_score * news_X_news_input.view(B, 2, D), dim=1)
            
    def forward(self, batch_datas):
        """@params:
        uids: [B]
        history_nids: [B, config.history_len]
        history_len: [B]
        category_ids: [B, config.history_len]
        topic_ids: [B, config.history_len]
        cate_news_samples: [B, config.history_len, config.sample_size]
        cate_news_sample_lens: [B, config.history_len]
        topic_news_samples: [B, config.history_len, config.sample_size]
        topic_news_sample_lens:[B, config.history_len]
        news_user_samples:[B, config.history_len, config.sample_size]
        news_user_sample_lens:[B, config.history_len]
        nids: [B]
        impr_category_id: [B]
        impr_topic_id: [B]
        impr_cate_news_sample: [B, config.sample_size]
        impr_cate_news_sample_len: [B]
        impr_topic_news_sample: [B, config.sample_size]
        impr_topic_news_sample_len:[B]
        impr_news_user_sample: [B, config.sample_size]
        impr_news_user_sample_len: [B]
        week: [B]
        hour: [B]
        relative_days: [B]
        relative_hours: [B]
        relative_seconds: [B]
        """
        uids, history_nids, history_len, category_ids, topic_ids, cate_news_samples, cate_news_sample_lens, \
            topic_news_samples, topic_news_sample_lens, news_user_samples, news_user_sample_lens, \
            nids, impr_category_id, impr_topic_id, impr_cate_news_sample, impr_cate_news_sample_len, impr_topic_news_sample, impr_topic_news_sample_len, \
            impr_news_user_sample, impr_news_user_sample_len, week, hour, relative_days, relative_hours, relative_seconds, labels = batch_datas
        uid_embedding = self.uid_embedding(uids)
        history_nids_embedding = self.nid_embedding_structual(history_nids) + self.nid_proj(self.nid_embedding_semantic_pre(history_nids))
        category_embedding = self.category_embedding(category_ids)
        topic_embedding = self.topic_embedding(topic_ids)
        cate_news_samples_embedding = self.nid_embedding_structual(cate_news_samples) + self.nid_proj(self.nid_embedding_semantic_pre(cate_news_samples))
        topic_news_samples_embedding = self.nid_embedding_structual(topic_news_samples) + self.nid_proj(self.nid_embedding_semantic_pre(topic_news_samples))
        news_user_samples_embedding = self.uid_embedding(news_user_samples)
        nid_embedding = self.nid_embedding_structual(nids) + self.nid_proj(self.nid_embedding_semantic_pre(nids))
        impr_category_id_embedding = self.category_embedding(impr_category_id)
        impr_topic_id_embedding = self.topic_embedding(impr_topic_id)
        impr_cate_news_sample_embedding = self.nid_embedding_structual(impr_cate_news_sample) + self.nid_proj(self.nid_embedding_semantic_pre(impr_cate_news_sample))
        impr_topic_news_sample_embedding = self.nid_embedding_structual(impr_topic_news_sample) + self.nid_proj(self.nid_embedding_semantic_pre(impr_topic_news_sample))
        impr_news_user_sample_embedding = self.uid_embedding(impr_news_user_sample)
        hour_embedding = self.hour_embedding(hour)
            
        B, L, S, D = cate_news_samples_embedding.size()
        
        # aggregate for user
        # meta-path News-Cate-News
        category_news_embedding = self.att_agg_user(uid_embedding, cate_news_samples_embedding, cate_news_sample_lens)
        news_cate_news_embedding = history_nids_embedding + category_embedding + category_news_embedding
        # news_cate_news_embedding = self.news_x_news_att(history_nids_embedding, category_embedding, category_news_embedding, self.news_cate_news_att)
        
        # meta-path News-Topic-News
        topic_news_embedding = self.att_agg_user(uid_embedding, topic_news_samples_embedding, topic_news_sample_lens)
        news_topic_news_embedding = history_nids_embedding + topic_embedding + topic_news_embedding
        # news_topic_news_embedding = self.news_x_news_att(history_nids_embedding, topic_embedding, topic_news_embedding, self.news_topic_news_att)
        
        # meta-path News-User
        user_embeddings = self.att_agg_user(uid_embedding, news_user_samples_embedding, news_user_sample_lens)
        news_user_embedding = history_nids_embedding + user_embeddings
        # news_user_embedding = self.news_X_att(history_nids_embedding, user_embeddings, self.news_user_att)        
        
        # aggregate for target nid
        # meta-path News-Cate-News
        impr_category_news_embedding = self.att_agg_news(nid_embedding, impr_cate_news_sample_embedding, impr_cate_news_sample_len)
        impr_news_cate_news_embedding = nid_embedding + impr_category_id_embedding + impr_category_news_embedding
        # impr_news_cate_news_embedding = self.news_x_news_att(nid_embedding, impr_category_id_embedding, impr_category_news_embedding, self.news_cate_news_att)
        
        # meta-path News-Topic-News
        impr_topic_news_embedding = self.att_agg_news(nid_embedding, impr_topic_news_sample_embedding, impr_topic_news_sample_len)
        impr_news_topic_news_embedding = nid_embedding + impr_topic_id_embedding + impr_topic_news_embedding
        # impr_news_topic_news_embedding = self.news_x_news_att(nid_embedding, impr_topic_id_embedding, impr_topic_news_embedding, self.news_topic_news_att)
        
        # meta-path News-User
        impr_user_embedding = self.att_agg_news(uid_embedding, impr_news_user_sample_embedding, impr_news_user_sample_len)
        impr_news_user_embedding = nid_embedding + impr_user_embedding
        # impr_news_user_embedding = self.news_X_att(nid_embedding, impr_user_embedding, self.news_user_att)
        
        # meta-path attention
        # B, L, D = news_cate_news_embedding.size()
        # meta_path_att_input = torch.cat([news_cate_news_embedding, news_topic_news_embedding, news_user_embedding], dim=-1)
        # meta_path_att_score = F.softmax(self.meta_path_attention(meta_path_att_input), dim=-1).view(B, L, 3, 1)
        # history_meta_path_embedding = torch.sum(meta_path_att_score * meta_path_att_input.view(B, L, 3, D), dim=2).squeeze()
        history_meta_path_embedding = torch.cat([news_cate_news_embedding, news_topic_news_embedding, news_user_embedding], dim=-1)
        
        # meta-path attention for impr
        # B, D = impr_news_cate_news_embedding.size()
        # impr_meta_path_att_input = torch.cat([impr_news_cate_news_embedding, impr_news_topic_news_embedding, impr_news_user_embedding], dim=-1)
        # impr_meta_path_att_score = F.softmax(self.meta_path_attention(impr_meta_path_att_input), dim=-1).view(B, 3, 1)
        # impr_meta_path_embedding = torch.sum(impr_meta_path_att_score * impr_meta_path_att_input.view(B, 3, D), dim=1).squeeze() # [B, D]
        impr_meta_path_embedding = torch.cat([impr_news_cate_news_embedding, impr_news_topic_news_embedding, impr_news_user_embedding], dim=-1)
        
        # din attention
        # impr_meta_path_embedding_view = impr_meta_path_embedding.view(B, 1, 3*D)
        # din_att_input = torch.cat([
        #     impr_meta_path_embedding_view.repeat(1, L, 1),
        #     history_meta_path_embedding
        #     ], dim=-1)
        # din_att_raw_score = self.din_attention(din_att_input)  # B, L, 1
        din_att_raw_score = torch.multiply(history_meta_path_embedding, impr_meta_path_embedding.view(B, 1, -1))
        mask = (self.L_tensor.view(1, -1) >=  history_len.view(B, 1)).view(B, L, 1)
        din_att_raw_score = din_att_raw_score.masked_fill(mask, -1e9)
        din_att_score = F.softmax(din_att_raw_score, dim=1)
        din_history_embedding = torch.sum(din_att_score * history_meta_path_embedding, dim=1) # [B, 3D]
        
        all_concat_input = torch.cat([
            uid_embedding,
            nid_embedding,
            uid_embedding * nid_embedding,
            din_history_embedding,
            impr_meta_path_embedding,
            din_history_embedding * impr_meta_path_embedding,
            hour_embedding,
            relative_days.view(-1, 1),
            relative_hours.view(-1, 1),
            relative_seconds.view(-1, 1)
            ], dim=-1)
        fc_out = self.fc(all_concat_input)  # B, 2
        loss = F.cross_entropy(fc_out, labels)
        return fc_out, loss

In [11]:
# coding: utf-8
#pip install prefetch_generator
import torch
from tqdm import tqdm
import time
import datetime
from datetime import timedelta
import random
import numpy as np
import pandas as pd
import json
from collections import defaultdict
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from prefetch_generator import BackgroundGenerator

import multiprocessing
# from pandarallel import pandarallel  # df.apply加速
# # Initialization
# pandarallel.initialize(progress_bar=True, nb_workers=4)


NEWS_PAD = 'NewsPAD'
CATEGORY_PAD = 'Cate_PAD'
CATEGORY_UNK = 'Cate_UNK'
USER_PAD = 'UID_PAD'
USER_UNK = 'UID_UNK'
TOPIC_PAD_ID = 200


class DataLoaderX(DataLoader):
    def __iter__(self):
        return BackgroundGenerator(super().__iter__())
    
def collate_fn(batch_data, device, multi_gpu=True):
    tensors = [torch.as_tensor([x[i] for x in batch_data]) for i in range(26)]
    if not multi_gpu:
        tensors = [tensor.to(device) for tensor in tensors]
    return tensors


def get_time_dif(start_time):
    """获取已使用时间"""
    end_time = time.time()
    time_dif = end_time - start_time
    return timedelta(seconds=int(round(time_dif)))

def load_json_object(json_path):
    with open(json_path, 'r') as f:
        obj = json.load(f)
    return obj

def load_news(news_path, news2idx, cate2idx):
    news_dict = {}
    with open(news_path, 'r') as f:
        for line in f:
            try:
                fields = line.split('\t')
                news_id, category, subcategory, topic = fields[0], fields[1], fields[2], fields[3]
                nid, cate_id, topic_id = news2idx[news_id], cate2idx[subcategory], int(topic)
                news_dict[nid] = [cate_id, topic_id]
            except Exception as e:
                import pdb; pdb.set_trace()
    return news_dict

def load_df(behavior_path, debug=False):
    columns = ['uid', 'impr_time', 'week', 'hour', 'relative_days', 'relative_hours', 'relative_seconds', 'history', 'cur_impr']
    df = pd.read_csv(behavior_path, sep='\t', names=columns)
    if debug:
        df = df.sample(10000, random_state=1)
    df.fillna('', inplace=True)
    df['impr_date'] = df.impr_time.apply(lambda x: x.split()[0])
    df['relative_days'] = df['relative_days'].apply(lambda x: x / 6.0)
    df['relative_hours'] = df['relative_hours'].apply(lambda x: x / (6.0 * 24))
    df['relative_seconds'] = df['relative_seconds'].apply(lambda x: x / (6.0 * 24 * 3600))
    return df   

class MINDDataset(Dataset):
    def __init__(self, datas):
        self.datas = datas
        self.length = len(datas) 
    
    def __getitem__(self, index):
        return self.datas[index]
    
    def __len__(self):
        return self.length    

class MyDataset(object):
    def __init__(self, config):
        self.config = config
        self.news2idx = load_json_object(config.news2idx_path)
        self.uid2idx = load_json_object(config.uid2idx_path)
        self.cate2idx = load_json_object(config.cate2idx_path)
        # cate2news = load_json_object(config.cate2news_path)
        # self.cate2news = {int(k): v for k, v in cate2news.items()}
        # topic2news = load_json_object(config.topic2news_path)
        # self.topic2news = {int(k): v for k, v in topic2news.items()}
        news2uid = load_json_object(config.news2uid_path)
        self.news2uid = {int(k): v for k, v in news2uid.items()}
        self.news_dict = load_news(config.news_path, self.news2idx, self.cate2idx)
        config.uid_num, config.nid_num, config.cate_num = len(self.uid2idx), len(self.news2idx), len(self.cate2idx)
        self.train_df = load_df(config.train_behavior_path, config.debug)
        self.val_df = load_df(config.val_behavior_path, config.debug)
        self.logging = config.logging
        self.cate2news = {}
        self.topic2news = {}
        with open(config.new_news_path, 'r') as f:
            new_news = f.readline()
        self.config.new_news_set = {self.news2idx[x] for x in new_news.split()}
        self.logging.info('news size %s, user size %s, cate size %s', len(self.news2idx), len(self.uid2idx), len(self.cate2idx))
        self.has_init = False
        
    def init_per_epoch(self):
        if self.has_init and not self.config.reinit_epoch:
            return
        self.has_init = True
        seed = int(time.time())
        random.seed(seed)
        np.random.seed(seed)
        cate2news = load_json_object(self.config.cate2news_path)
        topic2news = load_json_object(self.config.topic2news_path)
        for k, v in cate2news.items():
            random.shuffle(v)
            self.cate2news[int(k)] = v
        for k, v in topic2news.items():
            random.shuffle(v)
            self.topic2news[int(k)] = v
        # self.cate2news = {int(k):  for k, v in cate2news.items()}
        # self.topic2news = {int(k): list(set(np.random.choice(v, 1000, replace=True))) for k, v in topic2news.items()}
        self.train_iter = self.build_train_iter()
        self.val_iter, self.test_iter = self.build_val_test_iter()
        
    def sample_old(self, x_ids, x_news_dict, excludes, size, default):
        sample_datas = []
        sample_lens = []
        for x, ex in zip(x_ids, excludes):
            if x not in x_news_dict:
                one_sample = [default] * size
                one_sample_len = 0
            else:
                if len(x_news_dict[x]) <= size:
                    one_sample = [y for y in x_news_dict[x] if y != ex]
                    one_sample_len = len(one_sample)
                    one_sample.extend([default] * (size - one_sample_len))
                else:
                    one_sample = np.random.choice(x_news_dict[x], size=size+1, replace=False)
                    one_sample = [y for y in one_sample if y != ex]
                    one_sample = one_sample[:size]
                    one_sample_len = size
            sample_datas.append(one_sample)
            one_sample_len = max(1, one_sample_len)
            sample_lens.append(one_sample_len)
        return sample_datas, sample_lens
    
    def sample_new(self, x_ids, x_news_dict, excludes, size, default):
        sample_datas = []
        sample_lens = []
        for x, ex in zip(x_ids, excludes):
            if x not in x_news_dict:
                one_sample = [default] * size
                one_sample_len = 0
            else:
                if len(x_news_dict[x]) <= size:
                    one_sample = [y for y in x_news_dict[x] if y != ex]
                    one_sample_len = len(one_sample)
                    one_sample.extend([default] * (size - one_sample_len))
                else:
                    length = len(x_news_dict[x])
                    idxs = np.random.randint(0, length, size+10)
                    one_sample = [x_news_dict[x][y] for y in idxs if x_news_dict[x][y] != ex]
                    if len(one_sample) >= size:
                        one_sample = one_sample[:size]
                        one_sample_len = size
                    else:
                        one_sample_len = len(one_sample)
                        one_sample.extend([default] * (size - one_sample_len))
            sample_datas.append(one_sample)
            one_sample_len = max(1, one_sample_len)
            sample_lens.append(one_sample_len)
        return sample_datas, sample_lens
    
    def sample(self, x_ids, x_news_dict, excludes, size, default):
        sample_datas = []
        sample_lens = []
        for x, ex in zip(x_ids, excludes):
            if x not in x_news_dict:
                one_sample = [default] * size
                one_sample_len = 0
            else:
                end = max(0, len(x_news_dict[x]) - size - 1)
                random_start = random.randint(0, end)
                one_sample = [y for y in x_news_dict[x][random_start:random_start + size + 1] if y != ex]
                if len(one_sample) >= size:
                    one_sample = one_sample[:size]
                    one_sample_len = size
                else:
                    one_sample_len = len(one_sample)
                    one_sample.extend([default] * (size - len(one_sample)))
            sample_datas.append(one_sample)
            one_sample_len = max(1, one_sample_len)
            sample_lens.append(one_sample_len)
        return sample_datas, sample_lens
    
    def process_one_record(self, record, mode='train'):
        user, history, cur_impr, week, hour, relative_days, relative_hours, relative_seconds = record
        ret = []
        uid = self.uid2idx[user] if user in self.uid2idx else self.uid2idx[USER_UNK]
        history_nids = [self.news2idx[news] for news in history.split()]  # news must in news2idx
        news_pad_idx = self.news2idx[NEWS_PAD]
        user_pad_idx = self.uid2idx[USER_PAD]
        history_len = min(len(history_nids), self.config.history_len)
        history_len = max(1, history_len)
        if len(history_nids) < self.config.history_len:
            history_nids.extend([news_pad_idx] * (self.config.history_len - len(history_nids)))
        else:
            history_nids = history_nids[:self.config.history_len]
        category_ids = [self.news_dict[nid][0] if (nid != news_pad_idx and nid in self.news_dict) else self.cate2idx[CATEGORY_PAD] for nid in history_nids]
        topic_ids = [self.news_dict[nid][1] if (nid != news_pad_idx and nid in self.news_dict) else TOPIC_PAD_ID for nid in history_nids]

        cate_news_samples, cate_news_sample_lens = self.sample(category_ids, self.cate2news, history_nids, self.config.sample_size, news_pad_idx)
        topic_news_samples, topic_news_sample_lens = self.sample(topic_ids, self.topic2news, history_nids, self.config.sample_size, news_pad_idx)
        news_user_samples, news_user_sample_lens = self.sample(history_nids, self.news2uid, [uid] * len(history_nids), self.config.sample_size, user_pad_idx)
        
        imprs = cur_impr.split()
        pos_news = list(filter(lambda x: x[-1] == '1', imprs))
        pos_nids = [self.news2idx[x[:-2]] for x in pos_news]
        neg_news = list(filter(lambda x: x[-1] == '0', imprs))
        neg_nids = [self.news2idx[x[:-2]] for x in neg_news]

        expected_neg_num = max(1, len(pos_nids)) * self.config.neg_pos_ratio
        if len(neg_nids) > expected_neg_num and mode == 'train':  # 训练模式才需要负采样
            neg_nids = np.random.choice(neg_nids, size=expected_neg_num, replace=False).tolist()
        
        labels = [1] * len(pos_nids) + [0] * len(neg_nids)
        impr_nids = pos_nids + neg_nids
        impr_category_ids = [self.news_dict[nid][0] if nid in self.news_dict else self.cate2idx[CATEGORY_PAD] for nid in impr_nids]
        impr_topic_ids = [self.news_dict[nid][1] if nid in self.news_dict else TOPIC_PAD_ID for nid in impr_nids]

        impr_cate_news_samples, impr_cate_news_sample_lens = self.sample(impr_category_ids, self.cate2news, impr_nids, self.config.sample_size, news_pad_idx)
        impr_topic_news_samples, impr_topic_news_sample_lens = self.sample(impr_topic_ids, self.topic2news, impr_nids, self.config.sample_size, news_pad_idx)
        impr_news_user_samples, impr_news_user_sample_lens = self.sample(impr_nids, self.news2uid, [uid] * len(impr_nids), self.config.sample_size, user_pad_idx)
        
        for nid, category_id, topic_id, impr_cate_news_sample, impr_cate_news_sample_len, \
            impr_topic_news_sample, impr_topic_news_sample_len, impr_news_user_sample, impr_news_user_sample_len, label \
            in zip(impr_nids, impr_category_ids, impr_topic_ids, impr_cate_news_samples, impr_cate_news_sample_lens,
                    impr_topic_news_samples, impr_topic_news_sample_lens, impr_news_user_samples, impr_news_user_sample_lens, labels):
            ret.append(
                (
                    uid, history_nids, history_len, category_ids, topic_ids, cate_news_samples, cate_news_sample_lens, 
                    topic_news_samples, topic_news_sample_lens, news_user_samples, news_user_sample_lens,
                    nid, category_id, topic_id, impr_cate_news_sample, impr_cate_news_sample_len,
                    impr_topic_news_sample, impr_topic_news_sample_len, impr_news_user_sample, impr_news_user_sample_len,
                    week, hour, relative_days, relative_hours, relative_seconds, label
                )
            )
        return ret
        
    def build_train_iter(self):
        self.logging.info('start train data sampling...')
        seed = int(time.time())
        random.seed(seed)
        np.random.seed(seed)
        mode = 'train'
        start_time = time.time()
        df = self.train_df
        records = [record for record in zip(df.uid, df.history, df.cur_impr, df.week, df.hour, df.relative_days, df.relative_hours, df.relative_seconds)]
        # lazy_results = []
        # for record in tqdm(records):
        #     ret = dask.delayed(self.process_one_record)(record, mode)
        #     lazy_results.append(ret)
        # results = dask.compute(*lazy_results, scheduler='processes', num_workers=4)
        # datas = [x for y in results for x in y]
        datas = []
        for record in tqdm(records):
            ret = self.process_one_record(record, mode)
            datas.extend(ret)
        self.logging.info('build_train_iter done. train data size {}. time usage {}'.format(len(datas), get_time_dif(start_time)))
        random.shuffle(datas)
        dataset = MINDDataset(datas)
        data_iter = DataLoaderX(
            dataset, self.config.batch_size, shuffle=True, num_workers=4, pin_memory=True,
            collate_fn=lambda x: collate_fn(x, self.config.device, self.config.multi_gpu)
            )
        return data_iter
        
    def build_val_test_iter(self):
        self.logging.info('start val data sampling....')
        mode = 'val'
        start_time = time.time()
        df = self.val_df
        records = [record for record in zip(df.uid, df.history, df.cur_impr, df.week, df.hour, df.relative_days, df.relative_hours, df.relative_seconds)]
        # lazy_results = []
        # for record in tqdm(records):
        #     ret = dask.delayed(self.process_one_record)(record, mode)
        #     lazy_results.append(ret)
        # results = dask.compute(*lazy_results, scheduler='processes', num_workers=4)
        # datas = [x for y in results for x in y]
        datas = []
        for record in tqdm(records):
            ret = self.process_one_record(record, mode)
            datas.extend(ret)
        self.logging.info('val data sampling done. time usage {}'.format(get_time_dif(start_time)))

        # 划分验证集和测试集
        random.seed(2021)
        random.shuffle(datas)
        val_len = int(0.3 * len(datas))
        val_datas = datas[:val_len]
        test_datas = datas[val_len:]
        val_dataset = MINDDataset(val_datas)
        val_iter = DataLoaderX(
            val_dataset, self.config.batch_size, shuffle=False, num_workers=4, pin_memory=True,
            collate_fn=lambda x: collate_fn(x, self.config.device, self.config.multi_gpu)
            )
        test_dataset = MINDDataset(test_datas)
        test_iter = DataLoaderX(
            test_dataset, self.config.batch_size, shuffle=False, num_workers=2, pin_memory=True,
            collate_fn=lambda x: collate_fn(x, self.config.device, self.config.multi_gpu)
            ) 
        
        # 记录new_user and new news idx
        self.config.new_user_index = {i for i, x in enumerate(test_datas) if x[0] == self.uid2idx[USER_UNK]}
        self.config.new_news_index = {i for i, x in enumerate(test_datas) if x[11] in self.config.new_news_set}
        self.config.test_user_id = [x[0] for x in test_datas] # 用来计算gauc
        labels = [x[-1] for x in test_datas]
        pos_num, test_size = sum(labels), len(labels)
        pos_rate = float(pos_num) / test_size
        self.config.entropy = -pos_rate * np.log(pos_rate) - (1 - pos_rate) * np.log(1 - pos_rate)
        self.logging.info('build_val_test_iter done. val data size {}. test data size {}. pos_rate {:.3f}. time usage {}'.format(len(val_datas), len(test_datas), pos_rate, get_time_dif(start_time)))
        self.logging.info('new_user_index len %s', len(self.config.new_user_index))
        self.logging.info('new_news_index len %s', len(self.config.new_news_index))
        return val_iter, test_iter


In [ ]:

# device_cnt = torch.cuda.device_count()
# default_device = ','.join(map(str, range(device_cnt)))
parser = argparse.ArgumentParser()
parser.add_argument('--model', default='MPNRec',help='choice one: MPNRec, wide_deep, DIN, NRMS, MEIRec')
parser.add_argument('--lr', default=1e-3, type=float)
parser.add_argument('--sample_size', default=30, type=int)
parser.add_argument('--history_len', default=100, type=int)
parser.add_argument('--batch_size', default=256, type=int)
parser.add_argument('--neg_pos_ratio', default=5, type=int)
parser.add_argument('--num_epochs', default=5, type=int)
parser.add_argument('--batches_per_check', default=500, type=int)
parser.add_argument('--require_improvement', default=3000, type=int)
parser.add_argument('--text_len', default=128, type=int)
parser.add_argument('--agg_method', default='pooling', type=str) # pooling or self_attention
parser.add_argument('--text_encoding', default='bert', type=str)
parser.add_argument('--use_pretrain', action='store_true')
parser.add_argument('--debug', action='store_true')
parser.add_argument('--multi_gpu',default=False,action='store_true')
parser.add_argument('--device', type=str)
parser.add_argument('--prefix', type=str)
parser.add_argument('--reinit_epoch', action='store_true')
args = parser.parse_args(args=[])
print("ok1")

np.random.seed(1)
torch.manual_seed(1)
torch.cuda.manual_seed_all(1)
torch.backends.cudnn.deterministic = True  # 保证每次结果一样
#torch.cuda.set_per_process_memory_fraction(1, 0)
print("ok2")

start_time = time.time()
module = importlib.import_module(args.model + '.{}_model'.format(args.text_encoding))
config = module.Config(args)
logging = config.logging
print("ok3")

if args.model in ['wide_deep', 'DIN', 'NRMS']:
    util_module = importlib.import_module('common_utils')
else:
    util_module = importlib.import_module(args.model + '.utils')
dataset = util_module.MyDataset(config)
print("ok4")

model = module.Model(config)
#if config.multi_gpu:
#    model = nn.DataParallel(model)

logging.info(model)
model.to(config.device)
print("ok5")
# train
train(config, model, dataset)
logging.info('All Time usage: %s', get_time_dif(start_time))

ok1
ok2
ok3
ok4
ok5


100%|██████████| 596328/596328 [23:17<00:00, 426.66it/s]   
